# 第一阶段 步骤03：函数的连续调用

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 目标：从零构建深度学习框架 **DeZero** 的第三步。

---

## 核心目标

借助"函数的输入、输出都是 `Variable`"这一设计，把多个函数**串联**起来，完成更复杂的计算，并引出**计算图（computation graph）**的概念——这是后续反向传播的地基。

## 3.1 Exp 函数的实现

新增自然指数函数 `Exp`（计算 $e^x$）。实现方式和 `Square` 完全一样：继承 `Function`，在 `forward` 里写计算。

唯一的区别只是 `forward` 的内容从 `x ** 2` 换成了 `np.exp(x)`。

In [ ]:
import numpy as np

# 承接步骤02：Variable（数据的"箱子"）与 Function（通用基类）
class Variable:
    def __init__(self, data):
        self.data = data

class Function:
    def __call__(self, input):
        x = input.data          # 取出数据
        y = self.forward(x)     # 具体计算
        output = Variable(y)    # 包装成 Variable 返回
        return output

    def forward(self, x):
        raise NotImplementedError()

# 步骤02 已有的 Square
class Square(Function):
    def forward(self, x):
        return x ** 2

# 3.1 新增 Exp 函数：计算自然指数 e^x
class Exp(Function):
    def forward(self, x):
        return np.exp(x)

## 3.2 函数的连续调用

因为 `Function.__call__` 的输入和输出**都是 `Variable` 实例**，所以函数可以自然串联、连续调用。

例如计算 $y = (e^{x^2})^2$，即 `y = C(B(A(x)))`，其中 `A=Square`、`B=Exp`、`C=Square`。

In [ ]:
# 3.2 函数的连续调用：计算 y = (e^(x^2))^2
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)    # x^2       = 0.25
b = B(a)    # e^(x^2)   = e^0.25
y = C(b)    # (e^(x^2))^2

print(y.data)   # 1.648721270700128

## 计算图与复合函数

上面这一串计算，可以用**函数和变量交替排列的「计算图」**来表示：

```
x ──[A: Square]──▶ a ──[B: Exp]──▶ b ──[C: Square]──▶ y
```

- 圆框 = **变量**（`Variable`）
- 方框 = **函数**（`Function`）

把依次应用的多个函数看成"一个大的函数"，就叫**复合函数**。哪怕每个单独的函数都很简单（平方、指数），组合起来也能完成复杂计算。

## 这一步的"为什么"

用计算图表示计算，目的不是画图本身，而是**高效地求出每个变量的导数**——这个算法就是**反向传播（backpropagation）**。

为此，需要让变量和函数之间建立"谁创造了谁"的连接（后续会在 `Variable` 里记录 `creator`），从而能沿计算图反向遍历、用链式法则求导。

---

> 预告：步骤4 先用**数值微分**（中心差分近似）求出导数，作为理解"求导"的过渡；之后步骤5 才正式进入反向传播。